## Load dataset

In [5]:
import os 
import sys
sys.path.append('../Week_02_Vector_Search')

### Load dataset
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [7]:
documents[0].keys()

dict_keys(['content', 'filename'])

In [8]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions
Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

* 01-agentic-rag/lessons/01-intro.md
* 01-agentic-rag/lessons/02-environment.md
* 01-agentic-rag/lessons/03-rag.md

Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

What's the average number of input tokens across these 3 calls?

* 140
* 1400
* 14000
* 140000

In [13]:
for doc in documents[:3]:
    print(doc['filename'])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md


In [15]:
top3 = documents[:3]

In [17]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
openai_client = OpenAI()
openai_client

In [21]:
from evaluation_utils import llm_structured_retry
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [33]:
import numpy as np
np.mean([u.input_tokens for u in usages])

np.float64(1353.0)

## Load Ground Truth HW:

In [94]:
import pandas as pd
df_ground_truth = pd.read_csv('ground-truth_HW.csv')
ground_truth = df_ground_truth.to_dict(orient='records')
df_ground_truth 

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md
...,...,...
355,How should I break up a long article or transc...,07-project-example/lessons/07-chunking.md
356,"When I have several blog posts or wiki pages, ...",07-project-example/lessons/07-chunking.md
357,"For a single long PDF or video transcript, wha...",07-project-example/lessons/07-chunking.md
358,If I’m dealing with a book or other really lon...,07-project-example/lessons/07-chunking.md


In [41]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [44]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [45]:
chunks[0].keys()

dict_keys(['start', 'content', 'filename'])

## Q2. First result with text search

Take the first question from the ground truth:

```python
q = ground_truth[0]["question"]
```

After running `text_search` for it, what's the `filename` of the first result?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/13-function-calling.md`
* `01-agentic-rag/lessons/10-rag-next-steps.md`

In [71]:
from minsearch import Index
def text_index(documents):
    index = Index(text_fields = ['content'], keyword_fields = ['filename'])
    index.fit(documents)
    return index

def text_search(query, num_results = 5):
    results = text_idx.search(query = query, num_results=num_results)
    return results

In [95]:
q =  ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [96]:
chunk_index = text_index(chunks)
text_idx = chunk_index

text_search(q)[0]['filename']

'01-agentic-rag/lessons/03-rag.md'

## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

* `01-agentic-rag/lessons/01-intro.md`
* `01-agentic-rag/lessons/03-rag.md`
* `04-evaluation/lessons/11-evaluation-intro.md`
* `04-evaluation/lessons/12-rag-answers.md`

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.

In [145]:
import os
model_path = "../Week_02_Vector_Search/onnx_llm/models/Xenova/all-MiniLM-L6-v2"
from onnx_llm.embedder import Embedder
embed = Embedder(model_path)

from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields = ['filename'])

def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    results = vindex.search(query_vector= query_vector, num_results=num_results)
    return results

In [78]:
q_vector = embed.encode(q)

In [85]:
texts = [doc["content"] for doc in chunks]
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)
X.shape

  0%|          | 0/6 [00:00<?, ?it/s]

(295, 384)

In [146]:
vindex.fit(X, chunks) 
vector_search(q)[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

* 0.55
* 0.66
* 0.76
* 0.88

In [107]:
def compute_relevance_text(q):
    filename_truth = q['filename']
    results = text_search(q['question'])
    relevance = []
    for r in results:
        relevance.append(int(r['filename'] == filename_truth))
    return relevance

from tqdm.auto import tqdm
def compute_relevance_total_text(ground_truth):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)
    return relevance_total

In [108]:
relevance_total_text = compute_relevance_total_text(ground_truth)
relevance_total_text

  0%|          | 0/360 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [0, 0, 1, 1, 0],
 [0, 1, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [1, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [1, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1],
 [1, 1, 1, 1, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0,

In [111]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
    return round(cnt/len(relevance),2)

hit_rate(relevance_total_text)

0.76

## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the
module only evaluated keyword search.

What's the MRR?

* 0.35
* 0.45
* 0.55
* 0.65

In [118]:
def compute_relevance_vector(q):
    filename_truth = q['filename']
    results = vector_search(embed.encode(q['question']))
    relevance = []
    for r in results:
        relevance.append(int(r['filename'] == filename_truth))
    return relevance

from tqdm.auto import tqdm
def compute_relevance_total_vector(ground_truth):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance_vector(q)
        relevance_total.append(relevance)
    return relevance_total

In [119]:
relevance_total_vector = compute_relevance_total_vector(ground_truth)
relevance_total_vector

  0%|          | 0/360 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 1, 1, 1, 0],
 [0, 0, 1, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [1, 0, 1, 1, 0],
 [1, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 1],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 1, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 1, 0, 1, 0],
 [0, 0, 1, 1, 1],
 [1, 1, 1, 0, 1],
 [1, 1, 1, 1, 0],
 [1, 1, 0, 1, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0,

In [164]:
def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return round(total_score / len(relevance),2)
mrr(relevance_total_vector)

0.55

## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

* 1
* 50
* 100
* 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.

In [162]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

def compute_relevance_hybrid(q, k):
    filename_truth = q['filename']

    results = hybrid_search(q['question'], k)

    relevance = []

    for r in results:
        relevance.append(int(r['filename'] == filename_truth))

    return relevance

from tqdm.auto import tqdm

def compute_relevance_total_hybrid(ground_truth, k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_hybrid(q, k)
        relevance_total.append(relevance)

    return relevance_total

In [165]:
k_values = [1, 50, 100, 200]

for k in k_values:
    relevance_total = compute_relevance_total_hybrid(ground_truth, k)

    print(f'Tune k = {k}')
    print(f'MRR = {mrr(relevance_total):.2f}')

  0%|          | 0/360 [00:00<?, ?it/s]

Tune k = 1
MRR = 0.65


  0%|          | 0/360 [00:00<?, ?it/s]

Tune k = 50
MRR = 0.64


  0%|          | 0/360 [00:00<?, ?it/s]

Tune k = 100
MRR = 0.64


  0%|          | 0/360 [00:00<?, ?it/s]

Tune k = 200
MRR = 0.64
